In [ ]:
#This notebook is for general tesing and debugging of the code
#You might notice each block is a module from "modular" folder

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [8]:
#This is to test extraction and save everything for further testing
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tickers = ["AAPL.US","MSFT.US","ABDE.US","AMZN.US","PEP.US"]
end=datetime.today()
start=end-timedelta(days=2*365)
flag=True
for ticker in tickers:
    url=f"https://stooq.com/q/d/l/?s={ticker}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
    df_temp=(pd.read_csv(url,parse_dates=["Date"])
        .set_index("Date")
        .sort_index())
    df_temp.columns = [f"{ticker} Open", f"{ticker} High", f"{ticker} Low", f"{ticker} Close", f"{ticker} Volume"]
    if flag:
        data=df_temp
        flag =False
    else:
        data=data.join(df_temp)

#Defining the period is one of the most important things and key to everything workin propperly
period=pd.Timedelta("2W")
df=data

ValueError: Missing column provided to 'parse_dates': 'Date'

In [7]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tickers = ["AAPL.US","MSFT.US","ABDE.US","AMZN.US","PEP.US"]
end=datetime.today()
start=end-timedelta(days=2*365)
flag=True
for ticker in tickers:
    url = f"https://stooq.com/q/d/l/?s={ticker}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
    
    # 1. Cambiamos "Date" por "Date" (Mayúscula)
    # 2. Añadimos index_col=0 para que use la primera columna como índice directamente
    df_temp = pd.read_csv(url, parse_dates=["Date"]).set_index("Date").sort_index()

    # Importante: Stooq suele devolver: Date, Open, High, Low, Close, Volume
    # Si te da error de longitud, verifica el número de columnas con df_temp.shape[1]
    df_temp.columns = [f"{ticker} Open", f"{ticker} High", f"{ticker} Low", f"{ticker} Close", f"{ticker} Volume"]
    
    if flag:
        data = df_temp
        flag = False
    else:
        data = data.join(df_temp, how='outer') # 'outer' evita perder datos si las fechas no coinciden
period=pd.Timedelta("2W")
df=data


ValueError: Missing column provided to 'parse_dates': 'Date'

In [ ]:
import pandas as pd
import pandas_datareader.data as web
from datetime import datetime, timedelta

tickers = ["AAPL.US", "MSFT.US", "ADBE.US", "AMZN.US", "PEP.US"] # Nota: Corregido "ABDE" a "ADBE"
end = datetime.today()
start = end - timedelta(days=2*365)

# Descarga masiva: datareader acepta una lista y devuelve un DataFrame con MultiIndex
data = web.DataReader(tickers, 'stooq', start, end)

# El resultado viene con las fechas como índice y las columnas como MultiIndex:
# (Atributo, Ticker) -> Ejemplo: ('Close', 'AAPL.US')
# Para dejarlo con el formato de nombres que querías ("Ticker Atributo"):

data.columns = [f"{ticker} {col}" for col, ticker in data.columns]
data = data.sort_index()

print(data.head())
```

### ¿Por qué esta opción es mejor?
*   **Gestión de fechas automática**: No tienes que pelearte con si la columna se llama `Date`, `date` o `<DATE>`; el lector de Stooq de `pandas_datareader` ya sabe dónde buscar.
*   **Eficiencia**: Descarga todos los tickers en menos pasos y organiza los datos automáticamente en una sola tabla.
*   **Manejo de errores**: Si un ticker falla o no tiene datos en ese rango, es más fácil de depurar que con un bucle manual de `read_csv`.

**Nota importante sobre los Tickers**: En tu lista tenías `ABDE.US`, pero el ticker correcto para Adobe suele ser `ADBE.US`. Si el nombre es incorrecto, Stooq devolverá un error o un DataFrame vacío.

¿Te gustaría que añadamos alguna **métrica financiera** adicional (como retornos diarios) a esta tabla de datos?


In [9]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tickers = ["AAPL.US", "MSFT.US", "ADBE.US", "AMZN.US", "PEP.US"] # Corregido ADBE
end = datetime.today()
start = end - timedelta(days=2*365)
flag = True
data = pd.DataFrame()

for ticker in tickers:
    url = f"https://stooq.com{ticker}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
    
    # Solución: Usamos header=0 para identificar la primera fila como nombres
    # y luego forzamos el parseo de la primera columna (índice 0) como fecha
    df_temp = pd.read_csv(url)
    
    # Convertimos la primera columna a fecha manualmente para evitar errores de nombre
    df_temp.iloc[:, 0] = pd.to_datetime(df_temp.iloc[:, 0])
    df_temp = df_temp.set_index(df_temp.columns[0]).sort_index()
    
    # Renombrar columnas (Stooq suele traer Date, Open, High, Low, Close, Volume)
    df_temp.columns = [f"{ticker} Open", f"{ticker} High", f"{ticker} Low", f"{ticker} Close", f"{ticker} Volume"]
    
    if flag:
        data = df_temp
        flag = False
    else:
        # Usamos 'outer' para no perder filas si un ticker no operó un día
        data = data.join(df_temp, how='outer')

print(data.head())


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

In [ ]:
index="^NDX"
url=f"https://stooq.com/q/d/l/?s={index}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
data=(pd.read_csv(url,parse_dates=["Date"])
    .set_index("Date")
    .sort_index())
data.columns = [f"{index} Open", f"{index} High", f"{index} Low", f"{index} Close", f"{index} Volume"]

In [ ]:
index_name="^IPC"
index_v=index[f"{index_name} Close"].resample(period).mean()
    

In [ ]:
index_v

In [ ]:
data.to_csv("index_t.csv",index=True)

In [ ]:
df.to_csv("temporal.csv",index=True)

In [ ]:
df_read = pd.read_csv("temporal.csv", index_col=0, parse_dates=True)
index=pd.read_csv("index_t.csv", index_col=0, parse_dates=True)


In [ ]:
#esto da la matriz de correlaciones en array
tickers = ["AAPL.US","MSFT.US"]
flag=True
for ticker in tickers:
    df_temp=df_read[f"{ticker} Close"]
    df_temp=pd.DataFrame(df_temp)
    if flag:
        data=df_temp
        flag =False
    else:
        data=data.join(df_temp)
data=data.corr().to_numpy()


In [ ]:
period=pd.Timedelta("2W")
valores=df_read[f"{tickers[j]} Close"].resample(period).mean()


In [ ]:
np.array(valores)
valor=[valores,valores]
np.array(valor)

In [ ]:
periods=53
portfolio=[]
for j in range(len(tickers)):
    portfolio.append([])
    valores=df_read[f"{tickers[j]} Close"].resample(period).mean()
    for i in range(periods):
        portfolio[j].append(valores.iloc[i])
print(portfolio)

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([2, 3, 4, 5, 6])
c = np.array([5, 4, 3, 2, 1])
d = np.array([1,2,3,4,5])
# Stack the arrays vertically into a single 2D array, where each row is a variable
# np.vstack() or passing a list of arrays works
data_stacked = np.vstack([a, b, c,d])

# Calculate the correlation matrix using np.corrcoef()
# The default is rowvar=True, meaning each row is a variable, which is what we need here
correlation_matrix_np = np.corrcoef(data_stacked)

In [1]:
datos = [0]*5

datos[0]=[1,0.5,0,0,0]
datos[1]=[0.5,1,0,0,0]
datos[2]=[0,0,1,0.5,0.5]
datos[3]=[0,0,0.5,1,0]
datos[4]=[0,0,0.5,0,1]



In [34]:
flag=True
for i in range(len(datos)):
    if flag:
        nuevo=datos[i]
        flag =False
    else:
        nuevo=np.vstack([nuevo,datos[i]])
nuevo=np.corrcoef(nuevo)

In [35]:
nuevo

array([[ 1.        ,  0.6875    , -0.80178373, -0.5625    , -0.5625    ],
       [ 0.6875    ,  1.        , -0.80178373, -0.5625    , -0.5625    ],
       [-0.80178373, -0.80178373,  1.        ,  0.53452248,  0.53452248],
       [-0.5625    , -0.5625    ,  0.53452248,  1.        , -0.25      ],
       [-0.5625    , -0.5625    ,  0.53452248, -0.25      ,  1.        ]])

In [36]:
def metric_correlation_matrix(matrix):
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            matrix[i][j]=2*(1-matrix[i][j])
    print(matrix)
    matrix=np.sqrt(matrix)
    return matrix

In [37]:
metric_correlation_matrix(nuevo)

[[0.         0.625      3.60356745 3.125      3.125     ]
 [0.625      0.         3.60356745 3.125      3.125     ]
 [3.60356745 3.60356745 0.         0.93095503 0.93095503]
 [3.125      3.125      0.93095503 0.         2.5       ]
 [3.125      3.125      0.93095503 2.5        0.        ]]


array([[0.        , 0.79056942, 1.89830647, 1.76776695, 1.76776695],
       [0.79056942, 0.        , 1.89830647, 1.76776695, 1.76776695],
       [1.89830647, 1.89830647, 0.        , 0.96486011, 0.96486011],
       [1.76776695, 1.76776695, 0.96486011, 0.        , 1.58113883],
       [1.76776695, 1.76776695, 0.96486011, 1.58113883, 0.        ]])

In [16]:
flag=True
for i in range(4):
    if flag:
        nuevo=data_stacked[i]
        flag =False
    else:
        nuevo=np.vstack([nuevo,data_stacked[i]])

NameError: name 'data_stacked' is not defined

In [ ]:
correlation_matrix_np

In [ ]:
#Para volver métricos
for i in range(len(data)):
    for j in range(len(data)):
        data[i][j]=2*(1-data[i][j])
np.sqrt(data)

In [ ]:
#Testing benchmark.py
def amount_of_periods(period,end,start):
    #from inputs import start_date as start
    #from inputs import end_date as end
    #from main_module import start 
    #from main_module import end
    num_periods = (end - start) / period
    round_num_periods=(end - start)//period
    if num_periods!=round_num_periods:
        round_num_periods=round_num_periods+1
    print(round_num_periods)
    return round_num_periods

def calc_dev_by_period(df,companies, period):
    deviations_by_period=[]
    for company in companies:
        deviation=(df[f"{company} High"] - df[f"{company} Low"]).resample(period).std()
        deviations_by_period.append(deviation)
        print(len(deviations_by_period[0]))
    return deviations_by_period

def calc_vola(df, companies, period,end,start):
    deviations_by_period=calc_dev_by_period(df,companies, period)
    num_periods=amount_of_periods(period,end,start)
    volatility_weight=[]
    for i in range(num_periods):
        volatility_weight.append([])
        desv_sum=0
        for j in range(len(companies)):
            deviations_by_period[j][i]=1/deviations_by_period[j][i]
            desv_sum=desv_sum+deviations_by_period[j][i]
        for j in range(len(companies)):
            volatility_weight[i].append(deviations_by_period[j][i]/desv_sum)
    return volatility_weight

In [ ]:
def index_value(index, period, index_name):
    index_v=index[f"{index_name} Close"].resample(period).mean()
    return index_v

def index_returns(index,period,index_name,end,start):
    periods=amount_of_periods(period,end,start)
    index_v=index_value(index, period, index_name)
    index_r=[0]*periods
    for i in range(periods-1):
        index_r[i]=(index_v[i+1]/index_v[i])-1
    return index_r

In [ ]:
end=datetime.today()
start=end-timedelta(days=2*365)
index_name="^IPC"

In [ ]:
index_returns(index,period,index_name,end,start)

In [ ]:
benchmark=calc_vola(df,tickers,period,end,start)

In [ ]:
#Testing portfolio .py
def portfolio_value(benchmark, df, period, tickers):
    periods=amount_of_periods(period, end, start)
    portfolio=[0]*periods
    for j in range(len(tickers)):
        valores=df[f"{tickers[j]} Close"].resample(period).mean()
        for i in range(periods):
            portfolio[i]=portfolio[i]+benchmark[i][j]*valores[i]
    return portfolio

def portfolio_returns(portfolio,period,end,start):
    periods=amount_of_periods(period,end,start)
    port_return=[0]*periods
    for i in range(periods-1):
        port_return[i]=(portfolio[i+1]/portfolio[i])-1
    return port_return

In [ ]:
port=portfolio_value(calc_vola(df,tickers,period, end, start),df,period,tickers)

In [ ]:
r=portfolio_returns(port,period,end,start)
print(r)